# Biblioteca de IA — Equipo 2: Redes neuronales
Patrón **Strategy** · Combinación **Soft voting** · Métrica principal **ROC-AUC**

Ejecutar desde la carpeta que contiene `BibliotecaIA/`. Ver `README.md` para la documentación de clases.

## 1. Dataset y `Muestra`
Breast Cancer Wisconsin: 30 atributos numéricos continuos, clasificación binaria.

In [ ]:
import sys
from pathlib import Path

# Asegura que el directorio raíz del proyecto esté en PYTHONPATH
# para que Python pueda importar BibliotecaDeIAEquipo2
for candidate in [Path.cwd(), Path.cwd().parent, *Path.cwd().parents]:
    if (candidate / "BibliotecaDeIAEquipo2").exists():
        sys.path.insert(0, str(candidate))
        break

from BibliotecaDeIAEquipo2.datos.dataset import Dataset
ds = Dataset().cargar()
print(ds.resumen())
print(ds[0])
Xtr, Xte, ytr, yte = ds.dividir()

ModuleNotFoundError: No module named 'BibliotecaDeIAEquipo2'

## 2. Patrón Strategy
`RedNeuronal` es el contexto; cada configuración de MLP es una estrategia intercambiable. Así evitamos una clase por configuración o `if/else` dentro de la red.

In [ ]:
from BibliotecaDeIAEquipo2.modelos.perceptron_simple import PerceptronSimple
from BibliotecaDeIAEquipo2.modelos.red_neuronal import RedNeuronal
from BibliotecaDeIAEquipo2.patrones.strategy import *

perceptron = PerceptronSimple()
mlps = [RedNeuronal(EstrategiaMLP_2Capas()),
        RedNeuronal(EstrategiaMLP_3CapasTanh()),
        RedNeuronal(EstrategiaMLP_LogisticaSGD())]
variantes = [perceptron] + mlps
variantes

## 3. Polimorfismo
Mismo `entrenar()` / `predecir_proba()`, comportamiento distinto en cada variante.

In [ ]:
for v in variantes:
    v.entrenar(Xtr, ytr)
    print(f"{type(v).__name__:16s} | {v.nombre:38s} | {v.tiempo_entrenamiento:.3f}s | P(clase 1) primera muestra: {v.predecir_proba(Xte[:1])[0,1]:.4f}")

### Intercambio de estrategia en ejecución

In [ ]:
red = RedNeuronal(EstrategiaMLP_2Capas())
print(red.nombre)
red.cambiar_estrategia(EstrategiaMLP_3CapasTanh())
print(red.nombre)

## 4. Combinación: soft voting
Promedio de probabilidades: p(clase) = Σ wᵢ·pᵢ(clase); clase final = argmax.

In [ ]:
from BibliotecaDeIAEquipo2.ensemble.combinador import CombinadorSoftVoting
soft_mlp = CombinadorSoftVoting(mlps, nombre="Soft Voting (3 MLP)").marcar_entrenado()
soft_todos = CombinadorSoftVoting(variantes, nombre="Soft Voting (Perceptron + 3 MLP)").marcar_entrenado()

## 5. Evaluación (individual vs. combinada)

In [ ]:
from BibliotecaDeIAEquipo2.resultados.resultado import Resultado
from BibliotecaDeIAEquipo2.resultados.evaluador import Evaluador

def evaluar(m):
    return Resultado(m.nombre, yte, m.predecir(Xte), m.predecir_proba(Xte)[:, 1], m.tiempo_entrenamiento)

ev = Evaluador()
resultados = [evaluar(m) for m in variantes + [soft_mlp, soft_todos]]
ev.tabla(resultados)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
plt.figure(figsize=(6,5))
for r in resultados:
    fpr, tpr, _ = roc_curve(r.y_real, r.y_proba)
    plt.plot(fpr, tpr, label=r.nombre_modelo)
plt.plot([0,1],[0,1],'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(fontsize=7); plt.title('Curvas ROC'); plt.show()
print(ev.matriz_confusion(resultados[-2]))

## 6. Conclusiones
- Todos los modelos superan 0.995 de ROC-AUC; el dataset es casi linealmente separable, por eso el perceptrón simple compite con los MLP.
- El soft voting mejora exactitud y F1 (≈0.986 / 0.989) al compensar errores distintos; el ROC-AUC casi no cambia por estar saturado.
- El perceptrón entrena mucho más rápido.
- Con 143 muestras de prueba las diferencias son de 1–3 muestras; conviene validación cruzada.
- Reutilizar scikit-learn por composición permitió enfocarnos en el diseño OO y el patrón.